# Solve the Lanchester population system with native HHL

Regenerate the source eight-step finite-difference model with six years between steps. Normalize and pad the forcing vector, build the complete 32-dimensional Hermitian embedding, and use six phase bits and all 64 reciprocal rotations. A native Python quantum function specializes the full unitary, QPE and rotation table.

The local simulator executes the HHL block once. After checking its clean workspace, pass its entire logical state, including all phase and indicator branches, into a second public FlagQuantum Circuit. Append the normalized direct-solve state and execute the two Hadamards and five controlled swaps. The SWAP stage produces 30,000 actual local draws. This staged simulation preserves the unmeasured HHL state and uses separate physical resource counts for the two stages. The indicator-conditioned SWAP estimate and the indicator-one/phase-zero statevector fidelity use distinct conditioning.

The plots compare the native signed amplitudes and population trajectories with a fresh direct solve. Run with FlagQuantum, Matplotlib and Jupyter installed.


In [ ]:
from importlib import import_module
from pathlib import Path

import torch

from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.main import (
    OUTPUT,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.model import (
    DT,
    PHASE_WORDS,
    N,
    build_model,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.plots import (
    save_matrix_views,
    save_populations,
    save_vectors,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.quantum import (
    SHOTS,
    sampled_swap,
)
from examples.synthesis.applications.physical_systems.hhl_lanchester.overview.workflow import (
    show_result,
    solve,
)


def show(path: Path) -> None:
    display = import_module("IPython.display")
    display.display(display.Image(filename=str(path)))


## Regenerate the population model

In [ ]:
model = build_model()
print("Source shape:", model.operator.shape, "sample spacing:", DT)


In [ ]:
show(
    save_matrix_views(
        (
            ("Original operator", model.operator),
            ("Forcing vector", model.forcing[:, None]),
        ),
        OUTPUT / "original-system.png",
    )
)


In [ ]:
direct_population = torch.linalg.solve(model.operator, model.forcing)


In [ ]:
show(
    save_vectors(
        (("Direct classical", direct_population),), OUTPUT / "classical-vector.png"
    )
)


In [ ]:
show(save_populations(direct_population, OUTPUT / "classical-populations.png"))


## Embed the system in a Hermitian matrix

In [ ]:
forcing_norm = model.forcing.norm()
print("Forcing norm:", float(forcing_norm))


In [ ]:
show(
    save_matrix_views(
        (
            ("Hermitian embedding", model.hermitian),
            ("Normalized padded vector", model.padded[:, None]),
        ),
        OUTPUT / "hermitian-system.png",
    )
)


In [ ]:
torch.testing.assert_close(model.rescaled, model.rescaled.T)
assert float(model.phase_eigenvalues.min()) >= 0
assert float(model.phase_eigenvalues.max()) < 1
show(
    save_matrix_views(
        (("Shifted and rescaled operator", model.rescaled),),
        OUTPUT / "shifted-system.png",
    )
)


## Native HHL and coherent SWAP test

In [ ]:
assert model.angles.numel() == PHASE_WORDS
print("Reciprocal rotation entries:", model.angles.numel())


In [ ]:
direct_embedding = torch.linalg.solve(model.hermitian, model.padded)
torch.testing.assert_close(direct_embedding[: 2 * N] * forcing_norm, direct_population)


In [ ]:
assert model.evolution.shape == (4 * N, 4 * N)
print("Fresh unitary shape:", model.evolution.shape)


In [ ]:
print("Source swap-test width bound:", 28, "source shots:", SHOTS)


In [ ]:
solution = solve(model)
print(
    "Native width and operation count:",
    solution.resources.n_qubits,
    solution.resources.n_operations,
)


In [ ]:
swap = sampled_swap(solution.logical, solution.reference)


In [ ]:
print("Sampled overlap after indicator postselection:", swap.root_overlap)


In [ ]:
print("Native coherent SWAP stage: five controlled swaps and two Hadamards")


## Read out the HHL solution

In [ ]:
print("Named program outputs:", tuple(solution.program.registers))


In [ ]:
assert solution.logical.shape == (2, 32, 64)


In [ ]:
print(
    "Basis words represented in the postselected solution:",
    solution.statevector.numel(),
)


In [ ]:
show(
    save_vectors(
        (
            ("Direct classical", solution.reference),
            ("Native HHL", solution.statevector),
        ),
        OUTPUT / "hhl-vector.png",
    )
)


In [ ]:
show(
    save_populations(
        direct_population,
        OUTPUT / "hhl-populations.png",
        solution.statevector * forcing_norm,
    )
)


In [ ]:
show_result(solution, swap)
